# max-back-tied-half — ex2: derive relu_back as maximum_back0 with y=0 (half-mass at the kink)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `max-back-tied-half`. Running the final beacon cell reports progress against the `Backprop: max_back with tied half-mass` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: max_back with tied half-mass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`max-back-tied-half`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "max-back-tied-half"
DD_SUBTOPIC = "Backprop: max_back with tied half-mass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ReLU as maximum_back with y=0 — quick refresher

`relu(x) = maximum(x, 0)`. The backward fn `relu_back` is therefore `maximum_back0` evaluated with `y = 0`. The tie-splitting convention from ex1 propagates: at the kink `x == 0`, the gradient is `0.5 * grad_out` (half-mass), not `0` and not `1`.

Comparison to torch:
- `torch.nn.functional.relu` uses **`grad_out * (x > 0)`** — strict inequality → 0 at the kink.
- Our `relu_back` (via `maximum_back0(grad_out, _, x, zeros_like(x))`) uses **`(x > 0) + 0.5 * (x == 0)`** → 0.5 at the kink.

Both are valid subgradients of `max(x, 0)` at zero. The half-mass convention is symmetric and conserves gradient mass across both sides of the kink — convenient for theoretical analysis. Real frameworks pick strict inequality because the kink almost never occurs in practice with floating-point inputs.

### Exercise 2 — derive relu_back as maximum_back0 with y=0 (half-mass at the kink)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the half-mass tie-splitting convention by specializing maximum_back0 to y=0 and verifying that the resulting relu_back produces 0.5 * grad_out at the kink x==0, in contrast to torch's strict-inequality convention.
> Keywords: relu-back, subgradient, half-mass, kink, specialization
> ```

**KCs targeted:** `max-back-tied-half`, `unbroadcast-pattern`

Implement `relu_back(grad_out, out, x)` as a SPECIALIZATION of `maximum_back0`. The forward is `relu(x) = maximum(x, 0)`, so the backward fn is:

```
relu_back(grad_out, out, x) = maximum_back0(grad_out, out, x, zeros_like(x))
```

Both pieces:

1. **`maximum_back0(grad_out, out, x, y)`** — your batch-4 ex1 result. The half-mass rule: `bool_sum_x = (x > y) + 0.5 * (x == y)`, then return `grad_out * bool_sum_x` (no unbroadcast — shapes match in this scalar-y case).
2. **`relu_back(grad_out, out, x)`** — single-line specialization calling `maximum_back0` with `y = t.zeros_like(x)`.

**The point of this drill.** ReLU is the most-used activation in deep learning, and its backward is a one-line spec of `maximum_back0`. The tie-splitting convention propagates: at `x == 0` (the ReLU kink), `bool_sum_x = (0 > 0) + 0.5*(0 == 0) = 0.5`, so `relu_back(grad_out, _, 0) == 0.5 * grad_out`. Compare:
- Torch's `nn.functional.relu` uses **`grad_out * (x > 0)`** — strict inequality → 0 at the kink.
- Ours uses **half-mass** → 0.5 at the kink.

Both are valid subgradients. The test pins down the half-mass behavior explicitly and confirms agreement with torch in the strict-positive / strict-negative regions where both conventions coincide.

Inputs are raw `torch.Tensor`. No autograd.

In [ ]:
def maximum_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # Half-mass: 1 where x > y, 0 where x < y, 0.5 at ties.
    bool_sum = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * bool_sum


def relu_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return maximum_back0(grad_out, out, x, t.zeros_like(x))


<details><summary>Solution</summary>

```python
def maximum_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # Half-mass: 1 where x > y, 0 where x < y, 0.5 at ties.
    bool_sum = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * bool_sum


def relu_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return maximum_back0(grad_out, out, x, t.zeros_like(x))
```

**Why half-mass at the kink is mathematically the 'right' answer.** The subgradient of `max(x, 0)` at `x = 0` is the interval `[0, 1]` — any value in there is a valid one-sided derivative. `0.5` is the midpoint and the only choice that's symmetric around the kink and conserves the gradient mass across both sides (the analogue of the `bool_sum_x + bool_sum_y == 1` invariant from `max_back`).

**Why torch picks strict-inequality (`x > 0`) instead.** Two practical reasons: (1) `x == 0` is vanishingly rare in floating-point arithmetic, so the two conventions disagree on a measure-zero set; (2) the strict version avoids a branchy `==` comparison in a hot kernel — `(x > 0)` compiles to a single comparison op, `(x > 0) + 0.5*(x == 0)` requires two. Performance, not correctness.

**Why the specialization saves work.** Without it, ReLU would need its own dedicated back fn duplicating the half-mass logic. By specializing `maximum_back0(_, _, x, zeros_like(x))`, the framework re-uses a single tested code path — a tiny but real lessons-from-software-engineering moment inside the autograd library.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()